# Zolai End-to-End Pipeline: Similarity, Loan Words, & LLM Correction
This notebook combines three critical steps of the Zolai dataset pipeline into a single sequential workflow:
1. **Similarity Scoring**: Evaluates back-translated dictionary entries.
2. **Loan Word Filtering**: Uses an LLM to rescue legitimate loan words before purging English bleed-over.
3. **LLM Correction & Review**: Fixes technical seed data grammar using an LLM and displays a review UI.

In [ ]:
# --- 0. Setup & Installations ---
!pip install -q sentence-transformers google-generativeai pandas

import os
import json
import time
import pandas as pd
from IPython.display import display, HTML

# Setup paths (assuming Kaggle environment)
DATA_DIR = "/kaggle/input/zolai-dataset-train/data"
RESOURCES_DIR = "/kaggle/input/zolai-dataset-train/resources"
OUTPUT_DIR = "/kaggle/working/"

# Check for API Key
import google.generativeai as genai
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("GEMINI_API_KEY")
    genai.configure(api_key=api_key)
    print("Gemini API configured successfully.")
    GEMINI_AVAILABLE = True
except Exception as e:
    print(f"Warning: Could not load GEMINI_API_KEY from secrets. Steps 2 & 3 will fail. {e}")
    GEMINI_AVAILABLE = False


## 1. Similarity Scoring
Scores back-translated sentences against the original English to ensure accuracy.

In [ ]:
from sentence_transformers import SentenceTransformer, util

print("Loading embedding model (all-MiniLM-L6-v2)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

BACKTRANS_FILE = os.path.join(DATA_DIR, "tongdot_dict_backtranslated.jsonl")
SCORED_FILE = os.path.join(OUTPUT_DIR, "tongdot_dict_scored.jsonl")

if os.path.exists(BACKTRANS_FILE):
    scored_count = 0
    flagged_count = 0
    with open(BACKTRANS_FILE, "r", encoding="utf-8") as fin, open(SCORED_FILE, "w", encoding="utf-8") as fout:
        for line in fin:
            if not line.strip(): continue
            item = json.loads(line)
            orig = item.get("english_definition", "")
            back = item.get("back_translation", "")
            
            if orig and back:
                emb1 = model.encode(orig, convert_to_tensor=True)
                emb2 = model.encode(back, convert_to_tensor=True)
                score = util.cos_sim(emb1, emb2).item()
                item["similarity_score"] = round(score, 4)
                
                if score < 0.85:
                    flagged_count += 1
                    item["flagged_for_review"] = True
                
                fout.write(json.dumps(item, ensure_ascii=False) + "\n")
                scored_count += 1
    print(f"Similarity Scoring Complete: {scored_count} sentences scored. {flagged_count} flagged for review.")
else:
    print(f"File {BACKTRANS_FILE} not found. Skipping Step 1.")


## 2. Loan Word Filtering
Uses Gemini to identify legitimate loan words from flagged English bleed-over.

In [ ]:
if GEMINI_AVAILABLE:
    FLAGGED_FILE = os.path.join(DATA_DIR, "flagged_foreign_words.json")
    VOCAB_FILE = os.path.join(DATA_DIR, "zolai_unified_vocabulary.json")
    LOAN_WORDS_FILE = os.path.join(OUTPUT_DIR, "legitimate_loan_words.json")
    PURE_VOCAB_FILE = os.path.join(OUTPUT_DIR, "zolai_unified_vocabulary_pure.json")

    system_prompt_loans = """You are a Zolai (Tedim Chin) language expert.
Identify legitimate loan words from a list of flagged foreign/English words found in a Zolai dataset.
Legitimate loan words: 'computer', 'motor', 'plastic', 'phone', 'internet', 'radio', 'tv', 'video', 'bus', 'car', 'bank', 'office', 'school', 'hospital'.
Return a JSON object with a single key "loan_words" containing a list of strings."""

    if os.path.exists(FLAGGED_FILE):
        with open(FLAGGED_FILE, "r", encoding="utf-8") as f:
            flagged_data = json.load(f)
        
        # Just process top 100 for this pipeline demo to save time/API costs
        sorted_words = sorted(flagged_data.keys(), key=lambda w: flagged_data[w].get("frequency", 0), reverse=True)[:100]
        
        print(f"Processing top {len(sorted_words)} flagged words for loan word extraction...")
        gemini_model = genai.GenerativeModel(
            model_name="gemini-1.5-flash",
            system_instruction=system_prompt_loans,
            generation_config={"response_mime_type": "application/json", "temperature": 0.1}
        )
        
        prompt = f"Analyze these words and return the legitimate Zolai loan words:\n{json.dumps(sorted_words)}"
        try:
            response = gemini_model.generate_content(prompt)
            legitimate_loans = json.loads(response.text).get("loan_words", [])
            print(f"Identified {len(legitimate_loans)} legitimate loan words: {legitimate_loans[:10]}")
            
            with open(LOAN_WORDS_FILE, "w", encoding="utf-8") as f:
                json.dump(legitimate_loans, f, ensure_ascii=False, indent=2)
                
            # Rebuild vocab
            if os.path.exists(VOCAB_FILE):
                with open(VOCAB_FILE, "r", encoding="utf-8") as f:
                    vocab = json.load(f)
                clean_vocab = {k: v for k, v in vocab.items() if k not in flagged_data or k in legitimate_loans}
                with open(PURE_VOCAB_FILE, "w", encoding="utf-8") as f:
                    json.dump(clean_vocab, f, ensure_ascii=False, indent=2)
                print(f"Saved pure vocabulary to {PURE_VOCAB_FILE}. New size: {len(clean_vocab)}")
        except Exception as e:
            print(f"Loan word extraction failed: {e}")
    else:
        print(f"File {FLAGGED_FILE} not found. Skipping Step 2.")


## 3. LLM Correction & Human Review
Corrects technical seed data grammar and provides a review UI.

In [ ]:
if GEMINI_AVAILABLE:
    PROMPTS_FILE = os.path.join(RESOURCES_DIR, "tech_seed_data_prompts.jsonl")
    FIXED_FILE = os.path.join(OUTPUT_DIR, "tech_seed_data_fixed.jsonl")
    SYS_PROMPT_FILE = os.path.join(RESOURCES_DIR, "zolai_system_prompt.txt")

    if os.path.exists(PROMPTS_FILE) and os.path.exists(SYS_PROMPT_FILE):
        with open(SYS_PROMPT_FILE, "r", encoding="utf-8") as f:
            system_prompt_corr = f.read()
            
        items = []
        with open(PROMPTS_FILE, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip(): items.append(json.loads(line))
                
        gemini_model_pro = genai.GenerativeModel(
            model_name="gemini-1.5-pro",
            system_instruction=system_prompt_corr,
            generation_config={"response_mime_type": "application/json", "temperature": 0.2}
        )
        
        # Process a small sample for the notebook
        LIMIT = 5
        print(f"Running correction on a sample of {LIMIT} items...")
        
        review_data = []
        with open(FIXED_FILE, "w", encoding="utf-8") as fout:
            for data in items[:LIMIT]:
                prompt = data.get("llm_correction_prompt", "")
                if not prompt: continue
                try:
                    response = gemini_model_pro.generate_content(prompt)
                    # Clean potential markdown wrapping
                    clean_str = response.text.strip().strip('```json').strip('```').strip()
                    correction_data = json.loads(clean_str)
                    
                    data["fixed_zolai"] = correction_data.get("corrected_text", "")
                    data["back_translation"] = correction_data.get("back_translation", "")
                    data["errors_found"] = correction_data.get("errors_found", [])
                    if "llm_correction_prompt" in data: del data["llm_correction_prompt"]
                    
                    fout.write(json.dumps(data, ensure_ascii=False) + "\n")
                    review_data.append(data)
                    print(f"Corrected: {data['fixed_zolai'][:50]}...")
                    time.sleep(2)
                except Exception as e:
                    print(f"Correction failed for item: {e}")
                    
        # --- Human Review UI ---
        print("\n=== Human Review ===")
        if review_data:
            for r in review_data:
                if isinstance(r.get('errors_found'), list):
                    r['errors_found'] = ", ".join(r['errors_found'])
            df = pd.DataFrame(review_data)
            display_cols = ['text', 'fixed_zolai', 'back_translation', 'errors_found']
            df = df[[c for c in display_cols if c in df.columns]]
            pd.set_option('display.max_colwidth', None)
            display(HTML(df.to_html(classes='table table-striped', escape=False)))
    else:
        print(f"Required files not found. Skipping Step 3.")
